# DATA 266 Lab 2 - Part 2: Fine-Tuning a Diffusion Model for Domain-Specific Image Generation

**Story of this notebook**

| Step | What happens | Lab task |
|---|---|---|
| 0 | Run information, installs, config | C1 |
| 1 | Load the captioned image dataset | 2.1 |
| 2 | Clean: remove corrupted and duplicate images, report counts | 2.1 |
| 3 | Split 90/10 with seed 266, resize to 512x512 | 2.1 |
| 4 | Visualize and describe the dataset | 2.1 |
| 5 | Fix the 20 evaluation prompts (before training) | 2.3 |
| 6 | Load Stable Diffusion 1.5 and pre-process (cache latents and text embeddings) | 2.2 |
| 7 | Define the two runs (same data, same step budget) | 2.2 |
| 8 | Train run 1 and run 2, loss curves, run table | 2.2 |
| 9 | Push both models to Hugging Face with a model card | 2.2 |
| 10 | Generate 40 images per model (base, run 1, run 2), grids, upload | 2.3 |
| 11 | Evaluate: KID, CLIP score, LPIPS diversity, GPU-hours | 2.4 |
| 12 | Human evaluation: 25 blinded pairs | 2.4 |
| 13 | Failure modes | 2.4 |
| 14 | Safety: license, NSFW filter, memorization check | C4 |
| 15 | Conclusion | C3 |

Method (lectures 8-9): the VAE and text encoder stay frozen; only low-rank adapters in the UNet's attention layers are trained
to predict the noise added to image latents (the usual denoising MSE loss).

## Step 0 - Run information, installs, config

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

In [ ]:
!pip -q install diffusers transformers accelerate peft datasets torchmetrics torch-fidelity lpips huggingface_hub
import importlib.metadata as im
FROZEN = [f"{p}=={im.version(p)}" for p in ["diffusers", "transformers", "accelerate", "peft", "datasets", "torchmetrics", "torch-fidelity", "lpips", "huggingface_hub", "torch", "torchvision"]]
print("\n".join(FROZEN))

In [ ]:
import os, io, json, time, math, hashlib, gc, glob
from pathlib import Path
from collections import Counter
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch.nn.functional as F

# ---------------- configuration (everything you may want to change is here) ----------------
TEAM         = "##"                                               # two-digit team number
DATASET_ID   = "reach-vb/pokemon-blip-captions"                   # alternative: "lambdalabs/naruto-blip-captions"
BASE_MODEL   = "stable-diffusion-v1-5/stable-diffusion-v1-5"
RES          = 512
RUNS = {                                                          # same data, same step budget; only the adapter differs
    "run1_lora_r4":  {"rank": 4,  "alpha": 4,  "dora": False},
    "run2_lora_r16": {"rank": 16, "alpha": 16, "dora": False},    # second LoRA rank (set "dora": True to make it a DoRA run)
}
MAX_STEPS, BATCH_SIZE, GRAD_ACCUM, LR = 1500, 2, 2, 1e-4          # effective batch 4
VAL_EVERY, CKPT_EVERY, VAL_N = 250, 500, 32                      # held-out loss interval, checkpoint interval, held-out images used
KID_EXTRA_N  = 60                                                 # extra test captions generated per model for a larger, supplementary KID (0 = off)
HUMAN_EVAL_RUN_VS_RUN = False                                     # True = also build 25 blinded pairs run 1 vs run 2 (optional, beyond the lab requirement)
QUICK_TEST   = False                                              # True = 30 steps and 2 prompts, only to check the code runs
GEN_STEPS, GUIDANCE, GEN_SEEDS = 30, 7.5, [0, 1]                   # identical sampler settings for all three models
HF_USER      = os.environ.get("HF_USER", "")                      # your Hugging Face user or organisation
HF_TOKEN     = os.environ.get("HF_TOKEN", "")                     # or add it as a Kaggle/Colab secret named HF_TOKEN
FINAL_RUN    = "auto"                                             # "auto" = the fixed rule in Step 11 (KID, then CLIP tie-break); or name a run

OUT   = Path("/kaggle/working/p2") if os.path.exists("/kaggle/working") else Path("../outputs")
DATA  = OUT / "data"; MODELS = OUT / "models"; GEN = OUT / "generated"; HUMAN = OUT / "human_eval"
for d in (DATA, MODELS, GEN, HUMAN): d.mkdir(parents=True, exist_ok=True)
PROMPT_DIR = next((Path(p) for p in ["../prompts", "prompts"] if os.path.isdir(p)), OUT)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if QUICK_TEST: MAX_STEPS, VAL_EVERY, CKPT_EVERY, KID_EXTRA_N = 30, 10, 20, 4
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        try:
            from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN") or ""
        except Exception: pass
open(OUT / "requirements_frozen.txt", "w").write("\n".join(FROZEN))
print({"dataset": DATASET_ID, "base": BASE_MODEL, "steps": MAX_STEPS, "device": DEVICE, "hf_push": bool(HF_TOKEN and HF_USER)})

## Step 1 - Load the data

In [ ]:
from datasets import load_dataset
raw = load_dataset(DATASET_ID, split="train")
IMG_COL = next(c for c in raw.column_names if "image" in c.lower())
TXT_COL = next(c for c in raw.column_names if c.lower() in ("text", "caption", "captions", "prompt"))
print(raw, "\nimage column:", IMG_COL, "| caption column:", TXT_COL)
raw_n = len(raw)

## Step 2 - Clean
* **Corrupted:** the image cannot be decoded, is smaller than 64 px on a side, or has an empty caption.
* **Duplicates:** exact copies (MD5 of the pixels) and near-copies. A near-copy must pass two tests: average hashes within 5 bits (candidate), then a mean pixel difference under 4% on 32x32 thumbnails (confirmation). The first occurrence is kept and every removal is logged to `duplicates_removed.csv` with a visual audit.

In [ ]:
def ahash(img, n=8):                                   # 64-bit average hash, used only to propose near-duplicate candidates
    g = np.asarray(img.convert("L").resize((n, n), Image.BILINEAR), dtype=np.float32)
    return (g > g.mean()).ravel()
def thumb(img): return np.asarray(img.convert("L").resize((32, 32), Image.BILINEAR), dtype=np.float32) / 255
HAMMING_MAX, MAE_MAX = 5, 0.04                         # candidate if hashes differ in <= 5 bits; duplicate if mean pixel difference < 4%

rows, removed, dup_log = [], Counter(), []
seen_md5, kept_hash, kept_thumb, kept_idx = {}, [], [], []
for i in range(raw_n):
    try:
        ex = raw[i]; img = ex[IMG_COL]; img.load(); img = img.convert("RGB"); cap = " ".join(str(ex[TXT_COL]).split())
        if min(img.size) < 64 or not cap: removed["corrupted_or_empty"] += 1; continue
    except Exception:
        removed["corrupted_or_empty"] += 1; continue
    m = hashlib.md5(img.tobytes()).hexdigest(); h, t = ahash(img), thumb(img)
    if m in seen_md5:
        removed["exact_duplicate"] += 1; dup_log.append({"removed_idx": i, "kept_idx": seen_md5[m], "type": "exact", "hamming": 0, "pixel_mae": 0.0}); continue
    if kept_hash:
        ham = (np.stack(kept_hash) != h).sum(1); cand = np.where(ham <= HAMMING_MAX)[0]
        mae = [float(np.abs(kept_thumb[j] - t).mean()) for j in cand]
        if len(cand) and min(mae) < MAE_MAX:                       # hash match alone is not enough: confirm on pixels
            j = cand[int(np.argmin(mae))]; removed["near_duplicate"] += 1
            dup_log.append({"removed_idx": i, "kept_idx": kept_idx[j], "type": "near", "hamming": int(ham[j]), "pixel_mae": round(min(mae), 4)}); continue
    seen_md5[m] = i; kept_hash.append(h); kept_thumb.append(t); kept_idx.append(i)
    rows.append({"idx": i, "caption": cap, "width": img.size[0], "height": img.size[1]})
meta = pd.DataFrame(rows)
clean_report = {"loaded": raw_n, **removed, "removed_total": sum(removed.values()), "kept": len(meta)}
json.dump(clean_report, open(OUT / "cleaning_report.json", "w"), indent=1); print(clean_report)
dup_df = pd.DataFrame(dup_log); dup_df.to_csv(OUT / "duplicates_removed.csv", index=False)
if len(dup_df):                                                     # audit: look at what was removed
    show_d = dup_df.head(6); fig, ax = plt.subplots(2, len(show_d), figsize=(2.6 * len(show_d), 5.4), squeeze=False)
    for j, r in enumerate(show_d.itertuples()):
        ax[0, j].imshow(raw[int(r.kept_idx)][IMG_COL]); ax[0, j].set_title(f"kept #{r.kept_idx}", fontsize=8); ax[0, j].axis("off")
        ax[1, j].imshow(raw[int(r.removed_idx)][IMG_COL]); ax[1, j].set_title(f"removed #{r.removed_idx} ({r.type})", fontsize=8); ax[1, j].axis("off")
    plt.suptitle("Removed duplicates (audit)"); plt.tight_layout(); plt.show()

## Step 3 - Split and resize
90% train / 10% test with seed 266. Test images are used **only** for evaluation. Images are resized on the short side and centre-cropped to 512x512.

In [ ]:
def to_square(img, res=RES):
    w, h = img.size; s = res / min(w, h)
    img = img.resize((max(res, round(w * s)), max(res, round(h * s))), Image.LANCZOS)
    w, h = img.size; l, t = (w - res) // 2, (h - res) // 2
    return img.crop((l, t, l + res, t + res))

perm = np.random.RandomState(SEED).permutation(len(meta))
n_test = round(0.10 * len(meta))
meta["split"] = "train"; meta.loc[meta.index[perm[:n_test]], "split"] = "test"
PREP_VERSION = "short-side-lanczos+center-crop-v1"                 # change this text if to_square() changes
PREP_FP = hashlib.md5(json.dumps([DATASET_ID, RES, PREP_VERSION, SEED, [int(i) for i in meta.idx], list(meta.split), sorted(seen_md5)]).encode()).hexdigest()[:12]
DATA = DATA / PREP_FP                                             # new dataset, resolution, split or preprocessing -> new folder, never stale PNGs
for split in ("train", "test"): (DATA / split).mkdir(parents=True, exist_ok=True)
paths = []
for r in meta.itertuples():
    p = DATA / r.split / f"{r.idx:05d}.png"
    if not p.exists(): to_square(raw[int(r.idx)][IMG_COL].convert("RGB")).save(p)
    paths.append(str(p))
meta["path"] = paths; meta.to_csv(OUT / "dataset_meta.csv", index=False)
train_df, test_df = meta[meta.split == "train"].reset_index(drop=True), meta[meta.split == "test"].reset_index(drop=True)
assert not set(train_df.idx) & set(test_df.idx)
print({"train": len(train_df), "test": len(test_df), "resolution": RES, "preprocessing_cache": str(DATA)})

## Step 4 - Visualize and describe the dataset

In [ ]:
fig, ax = plt.subplots(2, 8, figsize=(20, 5.6))
for a, r in zip(ax.ravel(), train_df.sample(16, random_state=SEED).itertuples()):
    a.imshow(Image.open(r.path)); a.set_title(r.caption[:38], fontsize=7); a.axis("off")
plt.suptitle("Training samples (after cleaning and resizing)"); plt.tight_layout(); plt.show()

STOP = set("a an the of with and on in is to at its it".split())
wc = Counter(w for c in meta.caption for w in c.lower().replace(",", " ").split() if w not in STOP)
fig, ax = plt.subplots(1, 4, figsize=(20, 3.6))
meta.caption.str.split().str.len().plot.hist(bins=20, ax=ax[0], title="Caption length (words)")
pd.Series(dict(wc.most_common(15))).iloc[::-1].plot.barh(ax=ax[1], title="Most frequent caption words")
ax[2].scatter(meta.width, meta.height, s=6); ax[2].set_title("Original image sizes"); ax[2].set_xlabel("width"); ax[2].set_ylabel("height")
meta.split.value_counts().plot.bar(ax=ax[3], title="Split sizes", rot=0)
plt.tight_layout(); plt.show()
stats = {"images": len(meta), "unique_captions": int(meta.caption.nunique()), "median_caption_words": float(meta.caption.str.split().str.len().median()),
         "top_words": [w for w, _ in wc.most_common(8)], "share_captions_starting_with_top_phrase": round(float(meta.caption.str.lower().str.startswith(" ".join(meta.caption.iloc[0].lower().split()[:2])).mean()), 3)}
print(stats)

**Dataset description (edit after reading the outputs above).**
The set contains character illustrations paired with one short caption each, written automatically by the BLIP captioning model.
Images are single centred characters on plain backgrounds in one consistent cartoon style.

Two limitations / biases to state in the report (check each against the plots):
1. **Captions are machine-written and generic.** They are short, share a small vocabulary and repeat the same opening phrase, name few attributes, and can be wrong. The model therefore learns the *style* much better than fine-grained prompt control.
2. **Narrow visual distribution.** One art style, one subject per image, plain backgrounds, no scenes, no people or real-world objects. Expect weak results on out-of-domain and multi-object (compositional) prompts, and style "bleeding" into everything.

License: the characters are copyrighted, so the dataset and the fine-tuned models are for **non-commercial research only** (see Step 14).

## Step 5 - Fix the 20 evaluation prompts (before training)
10 in-domain prompts are test-set captions picked with seed 266; 5 out-of-domain and 5 compositional prompts come from `prompts/p2_prompts.json`.
The full list is written to `prompts/p2_prompts_final.json` - **commit it before training ends** (the Git timestamp is checked).

In [ ]:
fixed = json.load(open(PROMPT_DIR / "p2_prompts.json")) if (PROMPT_DIR / "p2_prompts.json").exists() else {
    "out_of_domain": ["a photo of a red sports car on a mountain road", "an astronaut riding a horse on the moon", "a bowl of ramen on a wooden table",
                      "a medieval castle at sunset", "a portrait of an old fisherman"],
    "compositional": ["two characters, one in red and one in blue", "a small green creature sitting on top of a large purple creature",
                      "three yellow creatures standing in a row", "a blue creature holding a red umbrella", "a creature with wings on the left and a creature with horns on the right"]}
in_domain = test_df.drop_duplicates("caption"); in_domain = in_domain.sample(min(10, len(in_domain)), random_state=SEED)
PROMPTS  = [{"id": f"ID{i+1:02d}", "category": "in_domain", "prompt": r.caption, "test_idx": int(r.idx)} for i, r in enumerate(in_domain.itertuples())]
PROMPTS += [{"id": f"OOD{i+1:02d}", "category": "out_of_domain", "prompt": p} for i, p in enumerate(fixed["out_of_domain"])]
PROMPTS += [{"id": f"CMP{i+1:02d}", "category": "compositional", "prompt": p} for i, p in enumerate(fixed["compositional"])]
assert len(PROMPTS) == 20, "need 10 distinct test captions"
json.dump(PROMPTS, open(PROMPT_DIR / "p2_prompts_final.json", "w"), indent=1)
if QUICK_TEST: PROMPTS = PROMPTS[:1] + PROMPTS[10:11]
pd.DataFrame(PROMPTS)

## Step 6 - Load Stable Diffusion 1.5 and pre-process
The VAE and the CLIP text encoder are frozen, so every training image is encoded **once** into latents and every caption into text embeddings.
Training then only needs the UNet on the GPU, which keeps memory low and steps fast.

In [ ]:
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, StableDiffusionPipeline, DPMSolverMultistepScheduler
from transformers import CLIPTokenizer, CLIPTextModel

tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer")
noise_sched = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

@torch.no_grad()
def build_cache(df):
    vae = AutoencoderKL.from_pretrained(BASE_MODEL, subfolder="vae").to(DEVICE).eval()
    te  = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder="text_encoder").to(DEVICE).eval()
    means, stds, embs = [], [], []
    for s in range(0, len(df), 16):
        b = df.iloc[s:s + 16]
        x = torch.stack([torch.from_numpy(np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 127.5 - 1).permute(2, 0, 1) for p in b.path]).to(DEVICE)
        d = vae.encode(x).latent_dist; means.append(d.mean.half().cpu()); stds.append(d.std.half().cpu())
        ids = tokenizer(list(b.caption), padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt").input_ids.to(DEVICE)
        embs.append(te(ids)[0].half().cpu())
    sf = vae.config.scaling_factor; del vae, te; gc.collect(); torch.cuda.empty_cache()
    return torch.cat(means), torch.cat(stds), torch.cat(embs), sf

LAT_MEAN, LAT_STD, TXT_EMB, SCALE = build_cache(train_df)
VAL_MEAN, _, VAL_EMB, _ = build_cache(test_df.iloc[:VAL_N])        # held-out images: loss monitoring only, never trained on
print("held-out latents:", tuple(VAL_MEAN.shape))
print("cached latents:", tuple(LAT_MEAN.shape), "| text embeddings:", tuple(TXT_EMB.shape), "| scaling factor:", SCALE)

# Pre-flight: make sure the generation pipeline has an NSFW safety checker now, not after an hour of training
from diffusers.pipelines.stable_diffusion import StableDiffusionSafetyChecker
from transformers import CLIPImageProcessor
def load_pipe():
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=dtype)
    if pipe.safety_checker is None:                                  # checkpoint ships without one: attach the standard checker explicitly
        pipe.safety_checker = StableDiffusionSafetyChecker.from_pretrained("CompVis/stable-diffusion-safety-checker", torch_dtype=dtype)
        pipe.feature_extractor = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
        pipe.register_to_config(requires_safety_checker=True)
    assert pipe.safety_checker is not None and pipe.feature_extractor is not None, "no NSFW safety checker available"
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config); pipe.set_progress_bar_config(disable=True)
    return pipe.to(DEVICE)
_p = load_pipe(); print("safety checker:", type(_p.safety_checker).__name__, "| scheduler:", type(_p.scheduler).__name__); del _p; gc.collect(); torch.cuda.empty_cache()

## Step 7 - Define the two runs
Both runs use the same training images, the same step budget, batch size, learning rate and seed. The only difference is the adapter (see `RUNS` in Step 0).
LoRA adds two small matrices (rank r) next to each attention projection in the UNet; only those are trained.

In [ ]:
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers
from diffusers.training_utils import cast_training_params

_h = hashlib.sha256()
for path_, cap_ in zip(train_df.path, train_df.caption): _h.update(open(path_, "rb").read()); _h.update(cap_.encode())
TRAIN_SHA = _h.hexdigest()

def fingerprint(cfg):
    """Everything that changes the trained adapter. A cached run is reused only if this matches."""
    key = {"cfg": cfg, "steps": MAX_STEPS, "bs": BATCH_SIZE, "accum": GRAD_ACCUM, "lr": LR, "seed": SEED, "base": BASE_MODEL,
           "dataset": DATASET_ID, "res": RES, "train_ids": hashlib.md5(",".join(map(str, train_df.idx)).encode()).hexdigest(),
           "train_content": TRAIN_SHA, "prep": PREP_FP}             # SHA-256 over the preprocessed training PNGs and their captions
    return hashlib.md5(json.dumps(key, sort_keys=True).encode()).hexdigest()[:12]

@torch.no_grad()
def heldout_loss(unet):
    """Denoising loss on held-out images with FIXED noise and timesteps, so values are comparable across steps and runs."""
    g = torch.Generator().manual_seed(1234); tot = 0.0; unet.eval()
    for s in range(0, len(VAL_MEAN), 8):
        lat = (VAL_MEAN[s:s + 8].float() * SCALE).to(DEVICE)
        noise = torch.randn(lat.shape, generator=g).to(DEVICE)
        t = torch.randint(0, noise_sched.config.num_train_timesteps, (len(lat),), generator=g).to(DEVICE)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            pred = unet(noise_sched.add_noise(lat, noise, t), t, encoder_hidden_states=VAL_EMB[s:s + 8].to(DEVICE)).sample
        tot += F.mse_loss(pred.float(), noise.float(), reduction="sum").item() / pred[0].numel()
    unet.train(); return tot / len(VAL_MEAN)

def save_adapter(unet, folder):
    folder.mkdir(parents=True, exist_ok=True)
    StableDiffusionPipeline.save_lora_weights(str(folder), unet_lora_layers=convert_state_dict_to_diffusers(get_peft_model_state_dict(unet)))

def train_run(name, rank, alpha, dora=False):
    torch.manual_seed(SEED); np.random.seed(SEED)
    unet = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder="unet", torch_dtype=torch.float16).to(DEVICE)
    unet.requires_grad_(False)
    unet.add_adapter(LoraConfig(r=rank, lora_alpha=alpha, init_lora_weights="gaussian", use_dora=dora,
                                target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
    cast_training_params(unet, dtype=torch.float32)                 # adapters in fp32, frozen weights in fp16
    params = [p for p in unet.parameters() if p.requires_grad]
    n_train, n_total = sum(p.numel() for p in params), sum(p.numel() for p in unet.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_STEPS)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
    g = torch.Generator().manual_seed(SEED); losses = []; val = {0: heldout_loss(unet)}; out = MODELS / name
    fp = fingerprint({"rank": rank, "alpha": alpha, "dora": dora}); state_f = out / "train_state.pt"; start, elapsed = 0, 0.0
    if state_f.exists():                                             # resume an interrupted run of the SAME configuration
        try: st = torch.load(state_f, map_location="cpu", weights_only=False)
        except Exception as e: st = {"fingerprint": None}; print(f"[{name}] checkpoint unreadable ({e}); starting from step 0")
        if st["fingerprint"] == fp:
            with torch.no_grad():
                for p_, w in zip(params, st["params"]): p_.copy_(w.to(p_.device, p_.dtype))
            opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"]); g.set_state(st["gen"])
            torch.set_rng_state(st["rng"]["torch"]); np.random.set_state(st["rng"]["numpy"]); random.setstate(st["rng"]["python"])
            if st["rng"]["cuda"] is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(st["rng"]["cuda"])
            start, losses, val, elapsed = st["step"], st["losses"], st["val"], st["elapsed"]; print(f"[{name}] resuming from step {start}")
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
    unet.train(); t0 = time.time()
    for step in range(start, MAX_STEPS):
        opt.zero_grad(set_to_none=True); step_loss = 0.0
        for _ in range(GRAD_ACCUM):
            idx = torch.randint(0, len(LAT_MEAN), (BATCH_SIZE,), generator=g)
            lat = ((LAT_MEAN[idx].float() + LAT_STD[idx].float() * torch.randn(LAT_MEAN[idx].shape, generator=g)) * SCALE).to(DEVICE)
            noise = torch.randn(lat.shape, generator=g).to(DEVICE)
            t = torch.randint(0, noise_sched.config.num_train_timesteps, (BATCH_SIZE,), generator=g).to(DEVICE)
            noisy = noise_sched.add_noise(lat, noise, t)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
                pred = unet(noisy, t, encoder_hidden_states=TXT_EMB[idx].to(DEVICE)).sample
            loss = F.mse_loss(pred.float(), noise.float()) / GRAD_ACCUM      # predict the added noise
            scaler.scale(loss).backward(); step_loss += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step(); losses.append(step_loss)
        if (step + 1) % VAL_EVERY == 0 or step + 1 == MAX_STEPS:
            val[step + 1] = heldout_loss(unet)
            print(f"[{name}] step {step + 1}/{MAX_STEPS}  train loss {np.mean(losses[-VAL_EVERY:]):.4f}  held-out loss {val[step + 1]:.4f}")
        if (step + 1) % CKPT_EVERY == 0 and step + 1 < MAX_STEPS:      # adapter + full training state: an interrupted session resumes here
            save_adapter(unet, out / f"checkpoint-{step + 1}")
            tmp_f = state_f.with_suffix(".tmp")
            torch.save({"fingerprint": fp, "step": step + 1, "params": [p_.detach().cpu() for p_ in params], "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "gen": g.get_state(),
                        "rng": {"torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                                "numpy": np.random.get_state(), "python": random.getstate()}, "losses": losses, "val": val, "elapsed": elapsed + time.time() - t0}, tmp_f)
            os.replace(tmp_f, state_f)                               # atomic: a crash mid-write leaves the previous checkpoint intact
    secs = elapsed + time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE == "cuda" else float("nan")
    save_adapter(unet, out)
    info = {"run": name, "method": ("DoRA" if dora else "LoRA") + f" r={rank}", "trainable_params": n_train, "trainable_%": round(100 * n_train / n_total, 3),
            "learning_rate": LR, "steps": MAX_STEPS, "batch_size": f"{BATCH_SIZE} x {GRAD_ACCUM} accum", "peak_gpu_gb": round(peak, 2),
            "train_minutes": round(secs / 60, 1), "gpu_hours": round(secs / 3600, 3), "final_train_loss": round(float(np.mean(losses[-100:])), 4),
            "heldout_loss_start": round(val[0], 4), "heldout_loss_end": round(val[max(val)], 4)}
    with open(out / "train_log.json", "w") as fh: json.dump({"info": info, "losses": losses, "val": val, "fingerprint": fp}, fh)
    assert (out / "pytorch_lora_weights.safetensors").stat().st_size > 0 and json.load(open(out / "train_log.json"))["fingerprint"] == fp
    if state_f.exists(): state_f.unlink()                            # only now: final adapter and log are verified on disk
    del unet, opt; gc.collect(); torch.cuda.empty_cache()
    return info, losses, val

## Step 8 - Train both runs
A finished run is reused only if its configuration fingerprint (adapter settings, steps, learning rate, batch, seed, dataset, training images) matches and its weights file exists; any change retrains. The adapter is also saved every `CKPT_EVERY` steps so an interrupted session keeps a usable checkpoint.

In [ ]:
RUN_INFO, LOSSES, VAL = {}, {}, {}
for name, cfg in RUNS.items():
    log, weights = MODELS / name / "train_log.json", MODELS / name / "pytorch_lora_weights.safetensors"
    d = json.load(open(log)) if log.exists() else {}
    if weights.exists() and d.get("fingerprint") == fingerprint(cfg):        # reuse only an identical configuration with its weights
        RUN_INFO[name], LOSSES[name], VAL[name] = d["info"], d["losses"], {int(k): v for k, v in d["val"].items()}; print("reloaded", name)
    else:
        RUN_INFO[name], LOSSES[name], VAL[name] = train_run(name, **cfg)
RUN_FP = "-".join(fingerprint(cfg) for cfg in RUNS.values())
run_table = pd.DataFrame(RUN_INFO.values()).set_index("run"); run_table.to_csv(OUT / "run_table.csv"); display(run_table)

fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
for name, l in LOSSES.items(): ax[0].plot(pd.Series(l).rolling(50, min_periods=1).mean(), label=name)
ax[0].set_title("Training loss (denoising MSE, 50-step moving average)"); ax[0].set_xlabel("step"); ax[0].legend()
for name, v in VAL.items(): ax[1].plot(list(v), list(v.values()), marker="o", label=name)
ax[1].set_title("Held-out loss (fixed noise and timesteps)"); ax[1].set_xlabel("step"); ax[1].legend(); plt.tight_layout(); plt.show()

Note on reading the curve: diffusion loss is noisy because each step draws random timesteps, and it falls only slightly during fine-tuning. A flat, low curve is normal. The held-out curve uses the same noise and timesteps every time, so it is smoother and comparable between runs: if it rises while the training loss falls, the adapter is overfitting. It is for monitoring only; no setting is chosen from it. Image quality is judged in Steps 10-12.

## Step 9 - Push both models to Hugging Face
Skipped (with a message) when `HF_USER` / `HF_TOKEN` are not set. The model card for the final run is written in Step 11, once the final run is chosen.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN) if (HF_TOKEN and HF_USER) else None
REPO = {name: f"{HF_USER}/data266-lab2-team{TEAM}-{name.replace('_', '-')}" for name in RUNS}
def push_model(name):
    if api is None: print("HF push skipped (set HF_USER and HF_TOKEN):", name); return
    api.create_repo(REPO[name], exist_ok=True); api.upload_folder(folder_path=str(MODELS / name), repo_id=REPO[name])
    print("pushed https://huggingface.co/" + REPO[name])
for name in RUNS: push_model(name)

## Step 10 - Generate images
Base model, run 1 and run 2 each generate the 20 prompts with seeds 0 and 1 (40 images per model), using the same sampler, steps and guidance.
The pipeline's built-in **NSFW safety checker stays on**; flagged images are counted for the safety section.

In [ ]:
pipe = load_pipe()                                                  # safety checker already verified in Step 6

import shutil
used = {p.get("test_idx") for p in PROMPTS}
KID_PROMPTS = test_df[~test_df.idx.isin(used)].drop_duplicates("caption").head(KID_EXTRA_N)      # extra test captions, for KID only
file_md5 = lambda f: hashlib.md5(open(f, "rb").read()).hexdigest()
GEN_FP = hashlib.md5(json.dumps([RUN_FP, GEN_STEPS, GUIDANCE, GEN_SEEDS, RES, [p["prompt"] for p in PROMPTS], list(KID_PROMPTS.caption), BASE_MODEL, FROZEN,
                                 [file_md5(MODELS / n / "pytorch_lora_weights.safetensors") for n in RUNS],             # the actual adapter bytes
                                 hashlib.md5("".join(file_md5(f) for f in test_df.path).encode()).hexdigest()]).encode()).hexdigest()[:12]
if (GEN / "fingerprint.txt").exists() and (GEN / "fingerprint.txt").read_text() != GEN_FP:
    shutil.rmtree(GEN); print("adapters or settings changed -> regenerating all images")       # never mix images from different runs
GEN.mkdir(exist_ok=True); (GEN / "fingerprint.txt").write_text(GEN_FP)
def valid_png(f):
    """A cached image is reused only if it opens fully and has the right size (an interrupted run can leave a broken file)."""
    try:
        with Image.open(f) as im: im.load(); return im.size == (RES, RES)
    except Exception: return False
MODEL_NAMES = ["base"] + list(RUNS)
records = []
for model_name in MODEL_NAMES:
    if model_name != "base": pipe.load_lora_weights(str(MODELS / model_name))
    (GEN / model_name).mkdir(exist_ok=True); t0 = time.time()
    for pr in PROMPTS:
        for seed in GEN_SEEDS:
            f = GEN / model_name / f"{pr['id']}_seed{seed}.png"; flag_f = f.with_suffix(".nsfw")
            if not valid_png(f):
                flag_f.unlink(missing_ok=True)
                out = pipe(pr["prompt"], num_inference_steps=GEN_STEPS, guidance_scale=GUIDANCE, height=RES, width=RES,
                           generator=torch.Generator(DEVICE).manual_seed(seed))
                out.images[0].save(f)
                if out.nsfw_content_detected and out.nsfw_content_detected[0]: flag_f.write_text("1")
            records.append({"file_name": f"{model_name}/{f.name}", "model": model_name, "prompt_id": pr["id"], "category": pr["category"],
                            "prompt": pr["prompt"], "seed": seed, "nsfw_flagged": flag_f.exists(), "path": str(f),
                            "sampler": "DPMSolverMultistep", "steps": GEN_STEPS, "guidance": GUIDANCE})
    (GEN / "kid_extra" / model_name).mkdir(parents=True, exist_ok=True)
    for r in KID_PROMPTS.itertuples():                                 # supplementary images: one seed per extra test caption
        f = GEN / "kid_extra" / model_name / f"{r.idx:05d}.png"
        if not valid_png(f):
            f.with_suffix(".nsfw").unlink(missing_ok=True)
            out = pipe(r.caption, num_inference_steps=GEN_STEPS, guidance_scale=GUIDANCE, height=RES, width=RES, generator=torch.Generator(DEVICE).manual_seed(0))
            out.images[0].save(f)
            if out.nsfw_content_detected and out.nsfw_content_detected[0]: f.with_suffix(".nsfw").write_text("1")
    if model_name != "base": pipe.unload_lora_weights()
    print(f"{model_name}: {len(PROMPTS) * len(GEN_SEEDS)} images in {time.time() - t0:.0f}s")
gen = pd.DataFrame(records)
gen.drop(columns="path").to_json(GEN / "metadata.jsonl", orient="records", lines=True)       # Hugging Face imagefolder metadata
NSFW_EXTRA = {m: len(list((GEN / "kid_extra" / m).glob("*.nsfw"))) for m in MODEL_NAMES}
print(gen.groupby("model").size().to_dict(), "| NSFW flagged (40 per model):", gen.groupby("model").nsfw_flagged.sum().to_dict(), "| NSFW flagged (extra KID set):", NSFW_EXTRA)
# Policy: a flagged image is blacked out by the checker. It stays in the 40-image set (counted, reported in Step 14) and is EXCLUDED from every KID.

In [ ]:
# Side-by-side grids for 8 prompts: 4 in-domain, 2 out-of-domain, 2 compositional (seed 0)
pick = [p for p in PROMPTS if p["category"] == "in_domain"][:4] + [p for p in PROMPTS if p["category"] == "out_of_domain"][:2] + [p for p in PROMPTS if p["category"] == "compositional"][:2]
fig, ax = plt.subplots(len(pick), len(MODEL_NAMES), figsize=(3.6 * len(MODEL_NAMES), 3.7 * len(pick)), squeeze=False)
for i, pr in enumerate(pick):
    for j, m in enumerate(MODEL_NAMES):
        ax[i, j].imshow(Image.open(GEN / m / f"{pr['id']}_seed0.png")); ax[i, j].axis("off")
        ax[i, j].set_title(f"{m}\n{pr['id']}: {pr['prompt'][:42]}", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "grid_base_vs_runs.png", dpi=110); plt.show()

### Step 10b - Upload the generated images to Hugging Face (dataset repository)
All 120 images (base, run 1, run 2) go to one Hugging Face **dataset** repo in `imagefolder` layout: `<model>/<prompt_id>_seed<k>.png` plus `metadata.jsonl` with `file_name`, model, prompt, category, seed, sampler settings and the NSFW flag.

In [ ]:
# ---- Upload ALL generated images to a Hugging Face DATASET repository (prompt, seed, model in metadata.jsonl) ----
bad = [f for f in gen.file_name if not valid_png(GEN / f)]            # validate first, upload second
assert not bad, f"missing or broken images listed in metadata: {bad[:3]}"
assert len(gen) == len(MODEL_NAMES) * len(PROMPTS) * len(GEN_SEEDS) and not gen.file_name.duplicated().any()
DATASET_REPO = f"{HF_USER}/data266-lab2-team{TEAM}-p2-generated"
if api is not None:
    api.create_repo(DATASET_REPO, repo_type="dataset", exist_ok=True)
    api.upload_folder(folder_path=str(GEN), repo_id=DATASET_REPO, repo_type="dataset", ignore_patterns=["*.nsfw", "kid_extra/*", "fingerprint.txt"])
    print("pushed https://huggingface.co/datasets/" + DATASET_REPO)
else: print("HF dataset upload skipped (set HF_USER and HF_TOKEN)")

In [ ]:
del pipe; gc.collect(); torch.cuda.empty_cache()

## Step 11 - Evaluation
| Measurement | Definition |
|---|---|
| **Primary: KID** | Kernel Inception Distance between the in-domain generations and the held-out test images. Lower is better. With only 20 generated images it is noisy, so the standard deviation over subsets is reported too. A **supplementary KID** on a larger set (the 20 plus extra test-caption generations) is shown beside it as a check on that noise. |
| **Primary: CLIP score** | 100 x cosine similarity between each prompt and its image (CLIP ViT-B/32). Higher = better prompt alignment. |
| **Secondary: diversity** | LPIPS distance between the two seeds of the same prompt, averaged. Higher = more varied outputs. |
| **Efficiency** | GPU-hours and peak GPU memory per training run (from Step 8). |

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from torchmetrics.image.kid import KernelInceptionDistance
import lpips

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
lp = lpips.LPIPS(net="alex").to(DEVICE).eval()

@torch.no_grad()
def clip_img(paths, bs=32):
    out = []
    for s in range(0, len(paths), bs):
        x = clip_proc(images=[Image.open(p).convert("RGB") for p in paths[s:s + bs]], return_tensors="pt").to(DEVICE)
        out.append(F.normalize(clip.get_image_features(**x), dim=-1).cpu())
    return torch.cat(out)
@torch.no_grad()
def clip_txt(texts, bs=32):
    out = []
    for s in range(0, len(texts), bs):
        x = clip_proc(text=texts[s:s + bs], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        out.append(F.normalize(clip.get_text_features(**x), dim=-1).cpu())
    return torch.cat(out)
def as_uint8(paths, size=299):
    return torch.stack([torch.from_numpy(np.asarray(Image.open(p).convert("RGB").resize((size, size)))).permute(2, 0, 1) for p in paths])
def as_lpips(p): return (torch.from_numpy(np.asarray(Image.open(p).convert("RGB").resize((256, 256)), dtype=np.float32)) / 127.5 - 1).permute(2, 0, 1)[None].to(DEVICE)

gen = gen.reset_index(drop=True)
img_f, txt_f = clip_img(list(gen.path)), clip_txt(list(gen.prompt))
assert len(img_f) == len(txt_f) == len(gen)                         # row i of both = row i of gen (same order, no shuffling)
gen["clip_score"] = (100 * (img_f * txt_f).sum(-1)).numpy()
def kid_update(metric, paths, is_real, bs=16):                     # batches: never the whole image set on the GPU at once
    for s in range(0, len(paths), bs): metric.update(as_uint8(paths[s:s + bs]).to(DEVICE), real=is_real)
KID_MIN = 8                                                        # fewer valid images than this: KID is not reported
def kid_of(fake_paths):
    if len(fake_paths) < KID_MIN or len(test_df) < KID_MIN:
        print(f"WARNING: only {len(fake_paths)} valid generated images - KID not computed"); return float("nan"), float("nan")
    k = KernelInceptionDistance(subset_size=min(len(fake_paths), len(test_df), 50), subsets=100).to(DEVICE)
    kid_update(k, list(test_df.path), True); kid_update(k, fake_paths, False)
    m_, s_ = (float(v) for v in k.compute()); del k; return m_, s_
rows = []
for m in MODEL_NAMES:
    g = gen[gen.model == m]; ind = g[g.category == "in_domain"]
    ind_ok = list(ind[~ind.nsfw_flagged].path)                        # safety-flagged (blacked-out) images are not scored
    k_mean, k_std = kid_of(ind_ok)                                     # required: the 20 in-domain images
    extra = sorted(str(p) for p in (GEN / "kid_extra" / m).glob("*.png") if not p.with_suffix(".nsfw").exists())
    kx_mean, kx_std = kid_of(ind_ok + extra) if extra else (float("nan"), float("nan"))   # supplementary: 20 + extra test captions
    with torch.no_grad():
        div = [float(lp(as_lpips(GEN / m / f"{p['id']}_seed{GEN_SEEDS[0]}.png"), as_lpips(GEN / m / f"{p['id']}_seed{GEN_SEEDS[1]}.png"))) for p in PROMPTS]
    gen.loc[g.index, "lpips_pair"] = [div[[p["id"] for p in PROMPTS].index(i)] for i in g.prompt_id]
    info = RUN_INFO.get(m, {})
    rows.append({"model": m, "KID in-domain (lower better)": round(k_mean, 4), "KID std": round(k_std, 4),
                 "KID n": len(ind_ok), f"KID larger set (n={len(ind_ok) + len(extra)})": round(kx_mean, 4), "KID larger set std": round(kx_std, 4),
                 "CLIP score (all 40)": round(g.clip_score.mean(), 2),
                 **{f"CLIP {c}": round(g[g.category == c].clip_score.mean(), 2) for c in ("in_domain", "out_of_domain", "compositional")},
                 "LPIPS diversity": round(float(np.mean(div)), 3), "Held-out loss": RUN_INFO.get(m, {}).get("heldout_loss_end", RUN_INFO[list(RUNS)[0]]["heldout_loss_start"]), "GPU-hours (train)": info.get("gpu_hours", 0), "Peak GPU GB (train)": info.get("peak_gpu_gb", 0)})
results = pd.DataFrame(rows).set_index("model"); results.to_csv(OUT / "results_table.csv"); gen.to_csv(OUT / "generation_scores.csv", index=False)
display(results)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
results["KID in-domain (lower better)"].plot.bar(ax=ax[0], yerr=results["KID std"], rot=0, title="KID vs test images (lower better)")
results[["CLIP in_domain", "CLIP out_of_domain", "CLIP compositional"]].plot.bar(ax=ax[1], rot=0, title="CLIP score by prompt type")
results["LPIPS diversity"].plot.bar(ax=ax[2], rot=0, title="Diversity between seeds (LPIPS)"); plt.tight_layout(); plt.show()

In [ ]:
# Choose the final run and write its model card
# Selection rule, fixed before looking at results:
#   1) lower larger-set KID wins (image quality / domain match);
#   2) if the two KIDs are within one standard deviation of each other, higher in-domain CLIP score wins (prompt alignment);
#   3) held-out loss and LPIPS diversity are reported for context only; the human evaluation then confirms or challenges the choice.
r2 = results.loc[list(RUNS)]; kid_col = next(c_ for c_ in r2.columns if c_.startswith("KID larger set (n="))
kcol, scol = (kid_col, "KID larger set std") if r2[kid_col].notna().all() else ("KID in-domain (lower better)", "KID std")
if r2[kcol].isna().any():                                           # KID unavailable for a run: fall back to prompt alignment, and say so
    best = worst = r2["CLIP in_domain"].idxmax(); tie = True; print("WARNING: KID missing for at least one run - selecting on in-domain CLIP score only")
else:
    best, worst = r2[kcol].idxmin(), r2[kcol].idxmax()
    tie = abs(r2.loc[best, kcol] - r2.loc[worst, kcol]) <= max(r2.loc[best, scol], r2.loc[worst, scol])
auto = r2["CLIP in_domain"].idxmax() if tie else best
print(f"KID used: {kcol} | difference within one std: {tie} | rule selects: {auto} ({'CLIP tie-break' if tie else 'lower KID'})")
display(r2[[kcol, scol, "CLIP in_domain", "Held-out loss", "LPIPS diversity"]])
if FINAL_RUN == "auto": FINAL_RUN = auto
other = [r for r in RUNS if r != FINAL_RUN][0]
print(f"FINAL_RUN = {FINAL_RUN}\n", results.loc[["base", FINAL_RUN, other]].T)
fi, fr = RUN_INFO[FINAL_RUN], results.loc[FINAL_RUN]
card = f"""---
license: other
license_name: research-only-non-commercial
base_model: {BASE_MODEL}
tags: [stable-diffusion, lora, text-to-image, diffusers]
---
# DATA 266 Lab 2 Part 2 - Team {TEAM} - {FINAL_RUN}

**Base model:** `{BASE_MODEL}` (CreativeML OpenRAIL-M). **Adapter:** {fi['method']} on the UNet attention projections.

**Data:** `{DATASET_ID}`, {len(train_df)} training images after removing {clean_report['removed_total']} corrupted/duplicate images; {len(test_df)} held out for evaluation (90/10 split, seed 266). 512x512, BLIP-generated captions.

**Training:** {fi['steps']} steps, batch {fi['batch_size']}, AdamW lr {fi['learning_rate']} cosine, fp16, {fi['trainable_params']:,} trainable parameters ({fi['trainable_%']}%), peak GPU {fi['peak_gpu_gb']} GB, {fi['train_minutes']} min ({fi['gpu_hours']} GPU-hours).

**Results (20 prompts x 2 seeds):** KID vs test images {fr['KID in-domain (lower better)']} (base {results.loc['base', 'KID in-domain (lower better)']}); CLIP score {fr['CLIP score (all 40)']} (base {results.loc['base', 'CLIP score (all 40)']}); LPIPS diversity {fr['LPIPS diversity']} (base {results.loc['base', 'LPIPS diversity']}).

**Limitations:** one narrow cartoon style; machine-written captions give weak fine-grained prompt control; out-of-domain and multi-object prompts are pulled toward the training style; KID is noisy with 20 images.

**License and use:** this card does not change any underlying license. The base model is under CreativeML OpenRAIL-M and the dataset's own terms apply; the training images depict copyrighted characters. This adapter is for **non-commercial research and teaching only**. Outputs were generated with the Stable Diffusion safety checker enabled.

```python
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained("{BASE_MODEL}").to("cuda")
pipe.load_lora_weights("{REPO[FINAL_RUN]}")
```
"""
open(MODELS / FINAL_RUN / "README.md", "w").write(card); push_model(FINAL_RUN); print(card[:600])
oi = RUN_INFO[other]                                              # the other run also gets a short card, so both repositories are documented
open(MODELS / other / "README.md", "w").write(f"""---
license: other
license_name: research-only-non-commercial
base_model: {BASE_MODEL}
tags: [stable-diffusion, lora, text-to-image, diffusers]
---
# DATA 266 Lab 2 Part 2 - Team {TEAM} - {other} (comparison run)

Comparison run, not the selected final model (final: `{REPO[FINAL_RUN]}`). {oi['method']}, {oi['steps']} steps, lr {oi['learning_rate']}, batch {oi['batch_size']},
{oi['trainable_params']:,} trainable parameters, peak GPU {oi['peak_gpu_gb']} GB, {oi['gpu_hours']} GPU-hours. Same data (`{DATASET_ID}`) and base model (`{BASE_MODEL}`) as the final run.
Training images depict copyrighted characters: non-commercial research and teaching only.
"""); push_model(other)

## Step 12 - Human evaluation (25 blinded pairs)
25 prompt/seed combinations are drawn with seed 266. Each pair shows the **final run** and the **base model** with left/right randomized and no labels.
At least 3 raters (both members plus one outside person) fill `human_eval/final_vs_base/ratings.csv`: for each pair, which side is better for **domain style** and for **prompt match** (`L` or `R`).
The answer key is saved separately in `pairs_key.csv` - do not show it to raters.

Optional: set `HUMAN_EVAL_RUN_VS_RUN = True` in Step 0 to also build 25 blinded pairs of run 1 against run 2 (`human_eval/run1_vs_run2/`). The lab only requires final run vs base.

In [ ]:
def make_pairs(model_a, model_b, tag, seed):
    """25 blinded pairs model_a vs model_b. Returns the key (which side model_a is on) and the folder."""
    folder = HUMAN / tag; folder.mkdir(exist_ok=True); rng = np.random.RandomState(seed)
    combos = [(p, s) for p in PROMPTS for s in GEN_SEEDS]
    key = []
    for k, i in enumerate(rng.permutation(len(combos))[:25], 1):
        p, s = combos[i]; a_left = bool(rng.rand() < 0.5); left, right = (model_a, model_b) if a_left else (model_b, model_a)
        sheet = Image.new("RGB", (2 * RES + 30, RES), "white")
        sheet.paste(Image.open(GEN / left / f"{p['id']}_seed{s}.png"), (0, 0)); sheet.paste(Image.open(GEN / right / f"{p['id']}_seed{s}.png"), (RES + 30, 0))
        sheet.save(folder / f"pair_{k:02d}.png")
        key.append({"pair": k, "prompt_id": p["id"], "prompt": p["prompt"], "seed": s, "a_side": "L" if a_left else "R"})
    key = pd.DataFrame(key); key.to_csv(folder / "pairs_key.csv", index=False); key[["pair", "prompt"]].to_csv(folder / "pairs_for_raters.csv", index=False)
    if not (folder / "ratings.csv").exists():
        pd.DataFrame([{"pair": k, "rater": r, "better_style": "", "better_prompt_match": ""} for r in ("member_A", "member_B", "outside_1") for k in key.pair]).to_csv(folder / "ratings_TEMPLATE.csv", index=False)
    return key, folder

def win_rate(key, folder, model_a, model_b):
    rp = folder / "ratings.csv"
    if not rp.exists(): print(f"{folder.name}: ratings.csv not found yet - fill ratings_TEMPLATE.csv and save it as ratings.csv"); return None
    r = pd.read_csv(rp, dtype={"better_style": str, "better_prompt_match": str}).merge(key[["pair", "a_side"]], on="pair")
    r["better_style"] = r.better_style.str.strip().str.upper(); r["better_prompt_match"] = r.better_prompt_match.str.strip().str.upper()
    r = r[r.better_style.isin(["L", "R"]) & r.better_prompt_match.isin(["L", "R"])]
    assert not r.duplicated(["rater", "pair"]).any(), "a rater scored the same pair twice"
    per = r.groupby("rater").pair.nunique(); incomplete = per[per < len(key)]
    assert incomplete.empty, f"every rater must score all {len(key)} pairs with L or R; incomplete: {incomplete.to_dict()}"
    assert r.rater.nunique() >= 3, "the lab needs at least 3 raters"
    res = {"comparison": f"{model_a} vs {model_b}", "raters": int(r.rater.nunique()), "ratings": len(r),
           f"{model_a} win rate (style)": round((r.better_style == r.a_side).mean(), 3), f"{model_a} win rate (prompt match)": round((r.better_prompt_match == r.a_side).mean(), 3)}
    print(res); display(r.assign(style_win=r.better_style == r.a_side, prompt_win=r.better_prompt_match == r.a_side).groupby("rater")[["style_win", "prompt_win"]].mean())
    return res

key_fb, dir_fb = make_pairs(FINAL_RUN, "base", "final_vs_base", SEED)             # required by the lab
human_final_vs_base = win_rate(key_fb, dir_fb, FINAL_RUN, "base")
if HUMAN_EVAL_RUN_VS_RUN:                                                         # optional: which adapter do people prefer?
    r1, r2_ = list(RUNS)
    key_rr, dir_rr = make_pairs(r1, r2_, "run1_vs_run2", SEED + 1); human_run_vs_run = win_rate(key_rr, dir_rr, r1, r2_)
print("pairs written under", HUMAN)

## Step 13 - Failure modes
Three candidates are pulled from the scores automatically; pick at least two for the report and describe what you see.
1. **Ignoring the prompt (style bleed):** out-of-domain prompts where the final run's CLIP score drops most against the base model.
2. **Repeated outputs (low diversity):** prompts where the two seeds of the final run are most alike (lowest LPIPS).
3. **Compositional errors:** compositional prompts with the lowest CLIP score (wrong count, colours swapped).

In [ ]:
piv = gen.pivot_table(index=["prompt_id", "category", "prompt"], columns="model", values="clip_score", aggfunc="mean").reset_index()
piv["drop_vs_base"] = piv["base"] - piv[FINAL_RUN]
f1 = piv[piv.category == "out_of_domain"].sort_values("drop_vs_base", ascending=False).head(2)
f2 = gen[gen.model == FINAL_RUN].drop_duplicates("prompt_id").sort_values("lpips_pair").head(2)
f3 = piv[piv.category == "compositional"].sort_values(FINAL_RUN).head(2)

def show(title, ids, cols):
    fig, ax = plt.subplots(len(ids), len(cols), figsize=(3.4 * len(cols), 3.6 * len(ids)), squeeze=False)
    for i, pid in enumerate(ids):
        prompt = next(p["prompt"] for p in PROMPTS if p["id"] == pid)
        for j, (m, s) in enumerate(cols):
            ax[i, j].imshow(Image.open(GEN / m / f"{pid}_seed{s}.png")); ax[i, j].axis("off"); ax[i, j].set_title(f"{m} seed {s}\n{prompt[:40]}", fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.savefig(OUT / (title.split(":")[0].replace(" ", "_").lower() + ".png"), dpi=100); plt.show()
show("Failure 1: prompt ignored on out-of-domain prompts", list(f1.prompt_id), [("base", 0), (FINAL_RUN, 0), (FINAL_RUN, 1)])
show("Failure 2: near-identical outputs across seeds", list(f2.prompt_id), [(FINAL_RUN, 0), (FINAL_RUN, 1)])
show("Failure 3: compositional prompts", list(f3.prompt_id), [("base", 0), (FINAL_RUN, 0), (other, 0)])
display(f1[["prompt", "base", FINAL_RUN, "drop_vs_base"]]); display(f2[["prompt", "lpips_pair"]]); display(f3[["prompt", "base", FINAL_RUN]])

## Step 14 - Safety (C4)
* **License:** the training images show copyrighted characters; base model is under CreativeML OpenRAIL-M. Models and generated images are for non-commercial research only (stated in the model card).
* **NSFW filter:** the Stable Diffusion safety checker was enabled for every generated image; the count of flagged images is printed below.
* **Memorization check (Carlini et al., 2023):** for the 10 in-domain images of the final run (seed 0), find the most similar **training** image by CLIP image similarity and show the 3 closest pairs. Near-copies would indicate memorized training data.

In [ ]:
print("NSFW-flagged images per model (40-image set):", gen.groupby("model").nsfw_flagged.sum().to_dict(), "| extra KID set:", NSFW_EXTRA)
ind = gen[(gen.model == FINAL_RUN) & (gen.category == "in_domain") & (gen.seed == GEN_SEEDS[0])].reset_index(drop=True)
sim = clip_img(list(ind.path)) @ clip_img(list(train_df.path)).T
best_sim, best_idx = sim.max(1)
mem = ind[["prompt_id", "prompt"]].assign(nearest_train=[train_df.path[int(i)] for i in best_idx], nearest_caption=[train_df.caption[int(i)] for i in best_idx], clip_similarity=best_sim.numpy().round(4))
mem.to_csv(OUT / "memorization_check.csv", index=False); display(mem.sort_values("clip_similarity", ascending=False))
top = mem.sort_values("clip_similarity", ascending=False).head(3).reset_index(drop=True)
fig, ax = plt.subplots(2, 3, figsize=(11, 7.6))
for j, r in top.iterrows():
    ax[0, j].imshow(Image.open(GEN / FINAL_RUN / f"{r.prompt_id}_seed{GEN_SEEDS[0]}.png")); ax[0, j].set_title(f"generated ({r.prompt_id})", fontsize=9); ax[0, j].axis("off")
    ax[1, j].imshow(Image.open(r.nearest_train)); ax[1, j].set_title(f"nearest training image\nCLIP sim {r.clip_similarity:.3f}", fontsize=9); ax[1, j].axis("off")
plt.suptitle("Memorization check: 3 closest generated/training pairs"); plt.tight_layout(); plt.savefig(OUT / "memorization_pairs.png", dpi=110); plt.show()
print("mean nearest-train similarity:", round(float(best_sim.mean()), 4), "| max:", round(float(best_sim.max()), 4))

## Step 15 - Conclusion
Fill in from the outputs above (every number must come from a cell output):

* **Did fine-tuning work?** KID of the final run against the base model (Step 11), and what the grids show (Step 10).
* **Prompt alignment:** CLIP score in-domain vs out-of-domain vs compositional; where the adapter helped and where it cost alignment.
* **Run 1 vs run 2:** quality, diversity, trainable parameters, GPU-hours and peak memory (Steps 8 and 11). **Which run we choose and why.**
* **Human evaluation:** win rate of the final run for style and for prompt match, with number of raters (Step 12).
* **Failure modes:** two of the three in Step 13, with the example images.
* **Safety:** license, NSFW flags, and whether the closest generated/training pairs look like copies (Step 14).
* **Limits:** 20 images make KID noisy; CLIP score is a weak judge of count and colour binding; one dataset, one seed for training.